# Frame Analysis: Rodin's Four-Slot Frame Engine Evaluation

This notebook analyzes data from children's interactions with the Rodin Frame Engine while creating mnemonics about microcontrollers.

## Research Questions

**RQ1:** To what extent did the frame support children's short-term and long-term learning through creating a mnemonic about the specified learning material?

**RQ2:** To what extent did the frame balance turns between students?

**RQ3:** To what extent did the frame accurately assess children's understanding?

**RQ4:** To what extent does children's prior knowledge (T1) predict the length and depth of the created mnemonic?

## Setup and Imports

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import json
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

## Data Loading and Preprocessing

In [4]:
# Define data paths
data_dir = Path('Data')
llm_data_dir = data_dir / 'LLM_interaction_Data'
qualtrics_dir = data_dir / 'Qualtrics'

# Load Qualtrics survey data
children_data = pd.read_csv(qualtrics_dir / 'BuildBots_children_T1T4T7.csv')
# engagement_data = pd.read_csv(qualtrics_dir / 'BuildBots_engagement_children_T1-T7.csv')
engagement_data = pd.read_csv(
    qualtrics_dir / 'BuildBots_engagement_children_T1-T7.csv', 
    encoding='ISO-8859-1'
)
print(f"Qualtrics Children Data Shape: {children_data.shape}")
print(f"Engagement Data Shape: {engagement_data.shape}")
print(f"\nQualtrics Children Data Columns (first 20):")
print(children_data.columns.tolist()[:20])

Qualtrics Children Data Shape: (85, 95)
Engagement Data Shape: (195, 34)

Qualtrics Children Data Columns (first 20):
['Duration (in seconds)', 'Timepoint', 'Participant ID', 'age', 'Gender', 'NAO_safe', 'NAO_competence', 'NAO_reliable', 'NAO_empathetic', 'NAO_scary', 'NAO_uncomfortable', 'NAO_social', 'Blossom_safe', 'Blossom_competent', 'Blossom_reliable', 'Blossom_empathetic', 'Blossom_scary', 'Blossom_uncomfortable', 'Blossom_social', 'Marty_safe']


In [5]:
# Function to load YAML session files
def load_yaml_file(filepath):
    """Load a YAML file and return its contents."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def load_all_sessions(base_dir):
    """Load all session YAML files from English and German folders."""
    sessions = []
    
    for language in ['English', 'German']:
        lang_dir = base_dir / language
        if lang_dir.exists():
            yaml_files = list(lang_dir.glob('*.yaml'))
            for yaml_file in yaml_files:
                # Skip prompt files
                if '_prompts' in yaml_file.name:
                    continue
                try:
                    session_data = load_yaml_file(yaml_file)
                    session_data['language'] = language
                    session_data['filename'] = yaml_file.name
                    sessions.append(session_data)
                except Exception as e:
                    print(f"Error loading {yaml_file}: {e}")
    
    return sessions

# Load all session data
sessions = load_all_sessions(llm_data_dir)
print(f"\nLoaded {len(sessions)} session files")


Loaded 0 session files


In [ ]:
# Display first session structure for exploration
if sessions:
    print("Sample session structure:")
    print(f"Session ID: {sessions[0].get('session_id')}")
    print(f"Students: {sessions[0].get('metadata', {}).get('students')}")
    print(f"Topic: {sessions[0].get('metadata', {}).get('topic')}")
    print(f"Number of events: {len(sessions[0].get('events', []))}")
    print(f"\nFrame memory keys: {sessions[0].get('frame_memory', {}).keys()}")

## Data Extraction Functions

In [ ]:
def extract_session_metrics(session):
    """Extract key metrics from a session for analysis."""
    metrics = {
        'session_id': session.get('session_id'),
        'language': session.get('language'),
        'students': session.get('metadata', {}).get('students', []),
        'n_students': len(session.get('metadata', {}).get('students', [])),
    }
    
    # Extract from frame_memory
    frame_memory = session.get('frame_memory', {})
    
    # Session-level metrics
    metrics['turn_count'] = frame_memory.get('turn_count', 0)
    metrics['session_phase'] = frame_memory.get('session_phase', 0)
    metrics['elapsed_time_minutes'] = frame_memory.get('elapsed_time_minutes', 0)
    
    # Mnemonic metrics
    mnemonic_state = frame_memory.get('mnemonic_state', {})
    metrics['concepts_selected'] = mnemonic_state.get('selected_concepts', [])
    metrics['n_concepts'] = len(mnemonic_state.get('selected_concepts', []))
    metrics['concepts_finalized'] = mnemonic_state.get('concepts_finalized', False)
    metrics['mnemonic_text'] = mnemonic_state.get('mnemonic_text', '')
    metrics['mnemonic_created'] = mnemonic_state.get('mnemonic_created', False)
    metrics['mnemonic_length'] = len(mnemonic_state.get('mnemonic_text', '').split())
    
    # Phase tracking
    metrics['reached_phase_3'] = frame_memory.get('session_phase', 0) >= 3
    
    # Participation metrics
    participation = frame_memory.get('balanced_turns_validator', {}).get('participation', {})
    for student, data in participation.items():
        metrics[f'{student}_contributions'] = data.get('contribution_count', 0)
        metrics[f'{student}_speaking_time'] = data.get('total_speaking_time_seconds', 0)
    
    # Comprehension tracking
    comprehension = frame_memory.get('comprehension_tracker', {})
    metrics['comprehension_concepts'] = comprehension.get('concepts', [])
    metrics['student_profiles'] = comprehension.get('by_student', {})
    
    return metrics

def extract_turn_level_data(session):
    """Extract turn-by-turn data from session events."""
    turns = []
    events = session.get('events', [])
    
    for event in events:
        # Look for analyze_input events from mnemonic_co_creator_marty
        if event.get('event') == '[analyze_input] mnemonic_co_creator_marty':
            data = event.get('data', {})
            turn_data = {
                'session_id': session.get('session_id'),
                'turn_count': data.get('turn_count'),
                'speaker': data.get('speaker'),
                'message': data.get('message'),
                'contribution_type': data.get('contribution_type'),
                'is_relevant': data.get('is_relevant'),
                'session_phase': data.get('session_phase'),
                'off_topic_duration': data.get('off_topic_duration', 0),
                'timestamp': event.get('timestamp')
            }
            turns.append(turn_data)
    
    return turns

def extract_comprehension_events(session):
    """Extract comprehension tracking events."""
    comprehension_events = []
    events = session.get('events', [])
    
    for event in events:
        if event.get('event') == '[analyze_input] comprehension_tracker_frame':
            data = event.get('data', {})
            if data.get('result') != 'no_output':
                comp_event = {
                    'session_id': session.get('session_id'),
                    'concepts': data.get('concepts', []),
                    'understood': data.get('per_turn_analysis', {}).get('understood', []),
                    'confused': data.get('per_turn_analysis', {}).get('confused', []),
                    'cumulative_assessments': data.get('cumulative_assessments_updated', []),
                    'timestamp': event.get('timestamp')
                }
                comprehension_events.append(comp_event)
    
    return comprehension_events

In [ ]:
# Extract metrics from all sessions
session_metrics_list = [extract_session_metrics(s) for s in sessions]
df_sessions = pd.DataFrame(session_metrics_list)

print(f"Session metrics dataframe shape: {df_sessions.shape}")
print(f"\nColumns: {df_sessions.columns.tolist()}")
df_sessions.head()

In [ ]:
# Extract turn-level data
all_turns = []
for session in sessions:
    all_turns.extend(extract_turn_level_data(session))

df_turns = pd.DataFrame(all_turns)
print(f"\nTurn-level data shape: {df_turns.shape}")
if not df_turns.empty:
    print(f"Columns: {df_turns.columns.tolist()}")
    df_turns.head()

In [ ]:
# Extract comprehension events
all_comprehension = []
for session in sessions:
    all_comprehension.extend(extract_comprehension_events(session))

df_comprehension = pd.DataFrame(all_comprehension)
print(f"\nComprehension events shape: {df_comprehension.shape}")
if not df_comprehension.empty:
    df_comprehension.head()

## RQ1: Learning Support through Mnemonic Creation

### Descriptive Statistics

In [ ]:
# RQ1 Descriptive: Mean number of concepts
print("="*60)
print("RQ1: LEARNING SUPPORT - DESCRIPTIVE STATISTICS")
print("="*60)

print("\n1. CONCEPT SELECTION")
print("-" * 40)
print(f"Mean number of concepts selected: {df_sessions['n_concepts'].mean():.2f} (SD={df_sessions['n_concepts'].std():.2f})")
print(f"Median: {df_sessions['n_concepts'].median():.0f}")
print(f"Range: {df_sessions['n_concepts'].min():.0f} - {df_sessions['n_concepts'].max():.0f}")

# Distribution of concepts
print(f"\nDistribution of concept counts:")
print(df_sessions['n_concepts'].value_counts().sort_index())

In [ ]:
# RQ1 Descriptive: Mean length of mnemonic
print("\n2. MNEMONIC LENGTH")
print("-" * 40)
# Only calculate for sessions where mnemonic was created
df_with_mnemonic = df_sessions[df_sessions['mnemonic_created'] == True]
print(f"Sessions with mnemonic created: {len(df_with_mnemonic)} / {len(df_sessions)}")

if len(df_with_mnemonic) > 0:
    print(f"Mean mnemonic length (words): {df_with_mnemonic['mnemonic_length'].mean():.2f} (SD={df_with_mnemonic['mnemonic_length'].std():.2f})")
    print(f"Median: {df_with_mnemonic['mnemonic_length'].median():.0f}")
    print(f"Range: {df_with_mnemonic['mnemonic_length'].min():.0f} - {df_with_mnemonic['mnemonic_length'].max():.0f}")

In [ ]:
# RQ1 Descriptive: Summary of contribution types
print("\n3. CONTRIBUTION TYPES")
print("-" * 40)
if not df_turns.empty:
    contribution_summary = df_turns['contribution_type'].value_counts()
    print("Distribution of contribution types:")
    for contrib_type, count in contribution_summary.items():
        percentage = (count / len(df_turns)) * 100
        print(f"  {contrib_type}: {count} ({percentage:.1f}%)")
    
    # Visualize
    plt.figure(figsize=(10, 6))
    contribution_summary.plot(kind='bar')
    plt.title('Distribution of Contribution Types')
    plt.xlabel('Contribution Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# RQ1 Descriptive: How many groups reached phase 3 (recall)
print("\n4. SESSION PHASES")
print("-" * 40)
reached_phase_3 = df_sessions['reached_phase_3'].sum()
total_sessions = len(df_sessions)
percentage_phase_3 = (reached_phase_3 / total_sessions) * 100

print(f"Groups that reached Phase 3 (recall): {reached_phase_3} / {total_sessions} ({percentage_phase_3:.1f}%)")
print(f"\nDistribution of final phases:")
print(df_sessions['session_phase'].value_counts().sort_index())

In [ ]:
# RQ1 Descriptive: Depth/difficulty level of mnemonics
# This requires coding the concepts based on difficulty
# Easy: what is it and what does it do
# Medium: how does it do it
# Difficult: precise vocabulary (3V, ESP32, HIGH/LOW)

def classify_concept_difficulty(concept):
    """Classify concepts by difficulty level."""
    concept_lower = str(concept).lower()
    
    # Difficult: precise technical terms
    difficult_terms = ['3v', 'esp32', 'high', 'low', 'c++', 'blockly', 'pins', 'spannung', 'voltage']
    if any(term in concept_lower for term in difficult_terms):
        return 'difficult'
    
    # Medium: functional descriptions
    medium_terms = ['steuern', 'control', 'programm', 'program', 'schalten', 'switch', 'ausgeben', 'output']
    if any(term in concept_lower for term in medium_terms):
        return 'medium'
    
    # Easy: basic definitions
    return 'easy'

print("\n5. MNEMONIC DEPTH/DIFFICULTY")
print("-" * 40)

# Calculate difficulty for each session
def calculate_session_depth(row):
    """Calculate depth score for a session based on concepts."""
    concepts = row['concepts_selected']
    if not concepts:
        return None, None, None
    
    difficulty_counts = {'easy': 0, 'medium': 0, 'difficult': 0}
    for concept in concepts:
        diff = classify_concept_difficulty(concept)
        difficulty_counts[diff] += 1
    
    # Calculate depth score (0-2 scale: 0=easy, 1=medium, 2=difficult)
    total = sum(difficulty_counts.values())
    if total == 0:
        return None, None, None
    
    depth_score = (
        0 * difficulty_counts['easy'] +
        1 * difficulty_counts['medium'] +
        2 * difficulty_counts['difficult']
    ) / total
    
    return depth_score, difficulty_counts['easy'], difficulty_counts['medium'], difficulty_counts['difficult']

df_sessions[['depth_score', 'n_easy', 'n_medium', 'n_difficult']] = df_sessions.apply(
    lambda row: pd.Series(calculate_session_depth(row)), axis=1
)

print(f"Mean depth score: {df_sessions['depth_score'].mean():.2f} (SD={df_sessions['depth_score'].std():.2f})")
print(f"\nDistribution of concept difficulty levels:")
print(f"  Easy concepts: {df_sessions['n_easy'].sum():.0f}")
print(f"  Medium concepts: {df_sessions['n_medium'].sum():.0f}")
print(f"  Difficult concepts: {df_sessions['n_difficult'].sum():.0f}")

In [ ]:
# RQ1 Descriptive: Co-creation level
# This requires analyzing who contributed concepts (students vs LLM)
# For now, we'll use a proxy based on contribution types

def calculate_cocreation_level(session_id, df_turns):
    """Calculate co-creation level for a session."""
    session_turns = df_turns[df_turns['session_id'] == session_id]
    if len(session_turns) == 0:
        return None
    
    # Count student-initiated concept contributions
    concept_contributions = session_turns[
        session_turns['contribution_type'].isin(['mnemonic_suggestion', 'builds_on_idea', 'concept_proposal'])
    ]
    
    if len(session_turns) == 0:
        return 0
    
    # Co-creation score: proportion of turns that are student concept contributions
    cocreation_score = len(concept_contributions) / len(session_turns)
    
    return cocreation_score

print("\n6. CO-CREATION LEVEL")
print("-" * 40)

if not df_turns.empty:
    df_sessions['cocreation_score'] = df_sessions['session_id'].apply(
        lambda sid: calculate_cocreation_level(sid, df_turns)
    )
    
    print(f"Mean co-creation score: {df_sessions['cocreation_score'].mean():.2f} (SD={df_sessions['cocreation_score'].std():.2f})")
    print(f"Range: {df_sessions['cocreation_score'].min():.2f} - {df_sessions['cocreation_score'].max():.2f}")
    print("\nNote: 0 = LLM did everything, 1 = students contributed all concepts")

In [ ]:
# RQ1 Descriptive: Off-topic contributions
print("\n7. OFF-TOPIC CONTRIBUTIONS")
print("-" * 40)

if not df_turns.empty:
    # Calculate off-topic contributions per session
    off_topic_by_session = df_turns[df_turns['contribution_type'] == 'off_topic'].groupby('session_id').size()
    df_sessions['n_off_topic'] = df_sessions['session_id'].map(off_topic_by_session).fillna(0)
    
    # Mean off-topic per student
    df_sessions['off_topic_per_student'] = df_sessions['n_off_topic'] / df_sessions['n_students']
    
    print(f"Mean off-topic contributions per student: {df_sessions['off_topic_per_student'].mean():.2f} (SD={df_sessions['off_topic_per_student'].std():.2f})")
    print(f"Total off-topic turns: {df_sessions['n_off_topic'].sum():.0f} / {len(df_turns)} ({(df_sessions['n_off_topic'].sum()/len(df_turns)*100):.1f}%)")
    
    # Transitions from off-topic to on-topic
    transitions = 0
    total_off_topic = 0
    
    for session_id in df_turns['session_id'].unique():
        session_turns = df_turns[df_turns['session_id'] == session_id].sort_values('turn_count')
        for i in range(len(session_turns) - 1):
            if (session_turns.iloc[i]['contribution_type'] == 'off_topic' and
                session_turns.iloc[i+1]['is_relevant'] == True):
                transitions += 1
                total_off_topic += 1
            elif session_turns.iloc[i]['contribution_type'] == 'off_topic':
                total_off_topic += 1
    
    if total_off_topic > 0:
        transition_rate = (transitions / total_off_topic) * 100
        print(f"\nTransitions from off-topic to on-topic: {transitions} / {total_off_topic} ({transition_rate:.1f}%)")

In [ ]:
# RQ1 Descriptive: Concepts from confusion to understanding
print("\n8. LEARNING TRAJECTORY (Confusion → Understanding)")
print("-" * 40)

if not df_comprehension.empty:
    # Track concepts that moved from confused to understood
    learning_trajectories = []
    
    for session_id in df_comprehension['session_id'].unique():
        session_comp = df_comprehension[df_comprehension['session_id'] == session_id].sort_values('timestamp')
        
        confused_concepts = set()
        learned_concepts = set()
        
        for _, event in session_comp.iterrows():
            # Track confused concepts
            for concept in event['confused']:
                confused_concepts.add(concept)
            
            # Check if previously confused concepts are now understood
            for concept in event['understood']:
                if concept in confused_concepts:
                    learned_concepts.add(concept)
        
        learning_trajectories.append({
            'session_id': session_id,
            'n_confused': len(confused_concepts),
            'n_learned': len(learned_concepts),
            'learning_rate': len(learned_concepts) / len(confused_concepts) if confused_concepts else 0
        })
    
    df_learning = pd.DataFrame(learning_trajectories)
    
    print(f"Total concepts flagged as confused/misconception: {df_learning['n_confused'].sum():.0f}")
    print(f"Concepts that moved to understood: {df_learning['n_learned'].sum():.0f}")
    print(f"Learning rate: {df_learning['n_learned'].sum() / df_learning['n_confused'].sum() * 100:.1f}%")
else:
    print("No comprehension tracking data available.")

### Significance Testing (RQ1)

Testing the relationship between mnemonic characteristics and learning outcomes.

In [ ]:
# Prepare data for significance testing
# We need to match session data with survey data (T7 and T8 microcontroller scores)

print("\n" + "="*60)
print("RQ1: SIGNIFICANCE TESTING")
print("="*60)

# First, explore the Qualtrics data to find relevant columns
print("\nExploring Qualtrics data for microcontroller knowledge measures...")
micro_cols = [col for col in children_data.columns if 'microcontroller' in col.lower()]
print(f"Microcontroller-related columns: {micro_cols}")

# Also check for T7 and T8 columns
t7_cols = [col for col in children_data.columns if 'T7' in col or 't7' in col]
t8_cols = [col for col in children_data.columns if 'T8' in col or 't8' in col]
print(f"\nT7 columns (first 10): {t7_cols[:10]}")
print(f"T8 columns (first 10): {t8_cols[:10]}")

In [ ]:
# Note: The actual statistical tests will depend on having participant IDs
# to match LLM session data with survey data
# For now, we'll prepare the framework for these analyses

print("\nNOTE: Statistical testing requires matching participant IDs between")
print("LLM session data and survey data. This will be implemented once")
print("the matching scheme is identified.")

# Placeholder for statistical tests
# These will test:
# 1. T7_microcontrollers ~ length_mnemonic + random
# 2. T7_microcontrollers ~ depth_mnemonic + random
# 3. T8_microcontrollers ~ length_mnemonic + random
# 4. T8_microcontrollers ~ depth_mnemonic + random

# Example framework (to be completed with actual data):
'''
from statsmodels.formula.api import mixedlm

# Short-term learning (T7)
model_t7_length = mixedlm("T7_microcontrollers ~ mnemonic_length", 
                          data=merged_data, 
                          groups=merged_data["participant_id"])
result_t7_length = model_t7_length.fit()
print(result_t7_length.summary())
'''

print("\nFramework for mixed-effects models prepared.")
print("Will test length and depth effects on short-term (T7) and long-term (T8) learning.")

## RQ2: Turn Balance Between Students

In [ ]:
print("\n" + "="*60)
print("RQ2: TURN BALANCE BETWEEN STUDENTS")
print("="*60)

# Calculate participation balance metrics
def calculate_balance_metrics(session):
    """Calculate balance metrics for a session."""
    students = session.get('metadata', {}).get('students', [])
    if len(students) < 2:
        return None
    
    frame_memory = session.get('frame_memory', {})
    participation = frame_memory.get('balanced_turns_validator', {}).get('participation', {})
    
    if not participation:
        return None
    
    contributions = [participation.get(student, {}).get('contribution_count', 0) for student in students]
    speaking_times = [participation.get(student, {}).get('total_speaking_time_seconds', 0) for student in students]
    
    if not contributions or sum(contributions) == 0:
        return None
    
    # Calculate Gini coefficient for contribution balance (0 = perfect equality, 1 = perfect inequality)
    def gini_coefficient(values):
        sorted_values = sorted(values)
        n = len(sorted_values)
        cumsum = np.cumsum(sorted_values)
        return (2 * np.sum((np.arange(n) + 1) * sorted_values)) / (n * cumsum[-1]) - (n + 1) / n if cumsum[-1] > 0 else 0
    
    return {
        'session_id': session.get('session_id'),
        'n_students': len(students),
        'mean_contributions': np.mean(contributions),
        'std_contributions': np.std(contributions),
        'gini_contributions': gini_coefficient(contributions),
        'mean_speaking_time': np.mean(speaking_times),
        'std_speaking_time': np.std(speaking_times),
        'gini_speaking_time': gini_coefficient(speaking_times),
        'min_contributions': min(contributions),
        'max_contributions': max(contributions)
    }

# Calculate balance for all sessions
balance_metrics = [calculate_balance_metrics(s) for s in sessions]
balance_metrics = [m for m in balance_metrics if m is not None]
df_balance = pd.DataFrame(balance_metrics)

print("\nPARTICIPATION BALANCE METRICS")
print("-" * 40)
print(f"Mean contributions per student: {df_balance['mean_contributions'].mean():.2f} (SD={df_balance['mean_contributions'].std():.2f})")
print(f"Mean Gini coefficient (contributions): {df_balance['gini_contributions'].mean():.3f} (SD={df_balance['gini_contributions'].std():.3f})")
print(f"  Note: Gini = 0 means perfect equality, 1 means perfect inequality")
print(f"\nMean speaking time per student: {df_balance['mean_speaking_time'].mean():.1f}s (SD={df_balance['mean_speaking_time'].std():.1f}s)")
print(f"Mean Gini coefficient (speaking time): {df_balance['gini_speaking_time'].mean():.3f} (SD={df_balance['gini_speaking_time'].std():.3f})")

In [ ]:
# Visualize participation balance
if not df_balance.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gini coefficient distribution
    axes[0].hist(df_balance['gini_contributions'], bins=15, edgecolor='black')
    axes[0].axvline(df_balance['gini_contributions'].mean(), color='red', linestyle='--', label='Mean')
    axes[0].set_xlabel('Gini Coefficient (Contributions)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Participation Balance\n(0=Perfect Equality)')
    axes[0].legend()
    
    # Contribution variability
    axes[1].scatter(df_balance['mean_contributions'], df_balance['std_contributions'], alpha=0.6)
    axes[1].set_xlabel('Mean Contributions per Student')
    axes[1].set_ylabel('Std Dev of Contributions')
    axes[1].set_title('Mean vs. Variability in Contributions')
    
    plt.tight_layout()
    plt.show()

## RQ3: Accuracy of Frame's Understanding Assessment

In [ ]:
print("\n" + "="*60)
print("RQ3: ACCURACY OF UNDERSTANDING ASSESSMENT")
print("="*60)

# Analyze comprehension assessments from frame
def summarize_comprehension(sessions):
    """Summarize understanding levels across all sessions."""
    understanding_levels = {'not_seen': 0, 'understood': 0, 'confused': 0, 'misconception': 0}
    
    for session in sessions:
        comprehension = session.get('frame_memory', {}).get('comprehension_tracker', {})
        student_profiles = comprehension.get('by_student', {})
        
        for student, profile in student_profiles.items():
            for concept, assessment in profile.items():
                level = assessment.get('level', '').lower()
                if level in understanding_levels:
                    understanding_levels[level] += 1
                elif 'understood' in level:
                    understanding_levels['understood'] += 1
                elif 'confused' in level:
                    understanding_levels['confused'] += 1
                elif 'misconception' in level:
                    understanding_levels['misconception'] += 1
    
    return understanding_levels

understanding_summary = summarize_comprehension(sessions)

print("\nUNDERSTANDING LEVELS ASSESSED BY FRAME")
print("-" * 40)
total_assessments = sum(understanding_summary.values())
if total_assessments > 0:
    for level, count in understanding_summary.items():
        percentage = (count / total_assessments) * 100
        print(f"{level.capitalize()}: {count} ({percentage:.1f}%)")
else:
    print("No understanding assessments found in frame memory.")
    print("\nUsing comprehension events from turn-by-turn data instead...")
    
    if not df_comprehension.empty:
        total_understood = sum(len(event['understood']) for _, event in df_comprehension.iterrows())
        total_confused = sum(len(event['confused']) for _, event in df_comprehension.iterrows())
        
        print(f"Concepts marked as understood: {total_understood}")
        print(f"Concepts marked as confused: {total_confused}")

In [ ]:
# RQ3 Significance Testing: Agreement with survey
print("\n" + "="*60)
print("RQ3: AGREEMENT WITH SURVEY (T7)")
print("="*60)

print("\nNOTE: This analysis requires:")
print("1. Matching participant IDs between LLM sessions and survey data")
print("2. Comparable understanding measures in both datasets")
print("\nOnce data is matched, we will calculate:")
print("- Cohen's Kappa for agreement on understanding levels")
print("- Pearson/Spearman correlation between frame and survey measures")
print("- Confusion matrix showing agreement patterns")

# Placeholder for agreement analysis
'''
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Example:
kappa = cohen_kappa_score(survey_understanding, frame_understanding)
correlation, p_value = pearsonr(survey_scores, frame_scores)
'''

## RQ4: Prior Knowledge Predicting Mnemonic Quality

In [ ]:
print("\n" + "="*60)
print("RQ4: PRIOR KNOWLEDGE EFFECTS")
print("="*60)

# Explore T1 knowledge variables
print("\nExploring T1 (baseline) knowledge measures...")
t1_cols = [col for col in children_data.columns if 'T1' in col or '_1' in col]
print(f"T1-related columns (first 20): {t1_cols[:20]}")

print("\nNOTE: This analysis requires:")
print("1. Matching participant IDs between sessions and survey")
print("2. T1 knowledge/prior knowledge measures from survey")
print("3. Mixed-effects models to account for nested data structure")

print("\nPlanned analyses:")
print("- Length ~ T1_knowledge + (1|child)")
print("- Depth ~ T1_knowledge + (1|child)")

# Placeholder for mixed-effects models
'''
from statsmodels.formula.api import mixedlm

# Model 1: Length prediction
model_length = mixedlm("mnemonic_length ~ T1_knowledge", 
                       data=merged_data, 
                       groups=merged_data["participant_id"])
result_length = model_length.fit()
print(result_length.summary())

# Model 2: Depth prediction
model_depth = mixedlm("depth_score ~ T1_knowledge", 
                      data=merged_data, 
                      groups=merged_data["participant_id"])
result_depth = model_depth.fit()
print(result_depth.summary())
'''

## Visualizations Summary

In [ ]:
# Create comprehensive visualization dashboard
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Number of concepts
ax1 = fig.add_subplot(gs[0, 0])
df_sessions['n_concepts'].hist(bins=10, ax=ax1, edgecolor='black')
ax1.set_xlabel('Number of Concepts')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Concept Counts')

# 2. Mnemonic length
ax2 = fig.add_subplot(gs[0, 1])
if len(df_with_mnemonic) > 0:
    df_with_mnemonic['mnemonic_length'].hist(bins=15, ax=ax2, edgecolor='black')
ax2.set_xlabel('Mnemonic Length (words)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Mnemonic Lengths')

# 3. Session phases
ax3 = fig.add_subplot(gs[0, 2])
phase_counts = df_sessions['session_phase'].value_counts().sort_index()
ax3.bar(phase_counts.index, phase_counts.values)
ax3.set_xlabel('Final Session Phase')
ax3.set_ylabel('Number of Sessions')
ax3.set_title('Distribution of Final Session Phases')

# 4. Depth scores
ax4 = fig.add_subplot(gs[1, 0])
df_sessions['depth_score'].dropna().hist(bins=10, ax=ax4, edgecolor='black')
ax4.set_xlabel('Depth Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Mnemonic Depth')

# 5. Co-creation scores
ax5 = fig.add_subplot(gs[1, 1])
if 'cocreation_score' in df_sessions.columns:
    df_sessions['cocreation_score'].dropna().hist(bins=10, ax=ax5, edgecolor='black')
ax5.set_xlabel('Co-creation Score')
ax5.set_ylabel('Frequency')
ax5.set_title('Student Co-creation Level')

# 6. Off-topic contributions
ax6 = fig.add_subplot(gs[1, 2])
if 'off_topic_per_student' in df_sessions.columns:
    df_sessions['off_topic_per_student'].hist(bins=10, ax=ax6, edgecolor='black')
ax6.set_xlabel('Off-topic per Student')
ax6.set_ylabel('Frequency')
ax6.set_title('Off-topic Contribution Rate')

# 7. Participation balance (Gini)
ax7 = fig.add_subplot(gs[2, 0])
if not df_balance.empty:
    df_balance['gini_contributions'].hist(bins=10, ax=ax7, edgecolor='black')
ax7.set_xlabel('Gini Coefficient')
ax7.set_ylabel('Frequency')
ax7.set_title('Participation Balance (0=Equal)')

# 8. Session duration
ax8 = fig.add_subplot(gs[2, 1])
df_sessions['elapsed_time_minutes'].hist(bins=15, ax=ax8, edgecolor='black')
ax8.set_xlabel('Session Duration (minutes)')
ax8.set_ylabel('Frequency')
ax8.set_title('Distribution of Session Durations')

# 9. Turn counts
ax9 = fig.add_subplot(gs[2, 2])
df_sessions['turn_count'].hist(bins=15, ax=ax9, edgecolor='black')
ax9.set_xlabel('Number of Turns')
ax9.set_ylabel('Frequency')
ax9.set_title('Distribution of Turn Counts')

plt.suptitle('Frame Analysis Overview Dashboard', fontsize=16, y=0.995)
plt.show()

## Summary Statistics Table

In [ ]:
# Create summary table for main metrics
summary_data = {
    'Metric': [
        'Number of sessions',
        'Mean concepts per mnemonic',
        'Mean mnemonic length (words)',
        'Mean depth score',
        'Sessions reaching phase 3 (%)',
        'Mean co-creation score',
        'Off-topic contributions (%)',
        'Mean contributions per student',
        'Mean Gini coefficient',
        'Mean session duration (min)',
    ],
    'Value': [
        len(df_sessions),
        f"{df_sessions['n_concepts'].mean():.2f} (SD={df_sessions['n_concepts'].std():.2f})",
        f"{df_with_mnemonic['mnemonic_length'].mean():.2f} (SD={df_with_mnemonic['mnemonic_length'].std():.2f})" if len(df_with_mnemonic) > 0 else 'N/A',
        f"{df_sessions['depth_score'].mean():.2f} (SD={df_sessions['depth_score'].std():.2f})",
        f"{(df_sessions['reached_phase_3'].sum() / len(df_sessions) * 100):.1f}%",
        f"{df_sessions['cocreation_score'].mean():.2f} (SD={df_sessions['cocreation_score'].std():.2f})" if 'cocreation_score' in df_sessions.columns else 'N/A',
        f"{(df_sessions['n_off_topic'].sum() / len(df_turns) * 100):.1f}%" if not df_turns.empty else 'N/A',
        f"{df_balance['mean_contributions'].mean():.2f} (SD={df_balance['mean_contributions'].std():.2f})" if not df_balance.empty else 'N/A',
        f"{df_balance['gini_contributions'].mean():.3f} (SD={df_balance['gini_contributions'].std():.3f})" if not df_balance.empty else 'N/A',
        f"{df_sessions['elapsed_time_minutes'].mean():.2f} (SD={df_sessions['elapsed_time_minutes'].std():.2f})",
    ]
}

summary_table = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("SUMMARY STATISTICS TABLE")
print("="*60)
print(summary_table.to_string(index=False))

## Export Data for Further Analysis

In [ ]:
# Save processed dataframes for use in statistical software or further analysis
output_dir = Path('Data/processed')
output_dir.mkdir(exist_ok=True)

# Save main dataframes
df_sessions.to_csv(output_dir / 'session_metrics.csv', index=False)
if not df_turns.empty:
    df_turns.to_csv(output_dir / 'turn_level_data.csv', index=False)
if not df_balance.empty:
    df_balance.to_csv(output_dir / 'balance_metrics.csv', index=False)
if not df_comprehension.empty:
    df_comprehension.to_csv(output_dir / 'comprehension_events.csv', index=False)

print("Processed data saved to:", output_dir)

## Next Steps

To complete the analysis, the following items need to be addressed:

1. **Participant Matching**: Establish a mapping between session IDs and participant IDs in the Qualtrics survey data

2. **Identify Survey Variables**: 
   - Locate T1 (baseline) knowledge measures for microcontrollers
   - Locate T7 (short-term) knowledge/understanding measures
   - Locate T8 (long-term) knowledge measures

3. **Complete Statistical Tests**:
   - Mixed-effects models for RQ1 (learning outcomes)
   - Agreement analysis for RQ3 (frame accuracy)
   - Prior knowledge prediction models for RQ4

4. **Additional Coding**:
   - Refine concept difficulty classification based on learning materials
   - Code contribution types for more nuanced co-creation analysis
   - Validate comprehension tracking categories

5. **Create Report**: Generate a comprehensive report with all findings suitable for publication